In [0]:
df = spark.read.table("`external-catalog`.default.employee_attrition")
display(df)



In [0]:
high_risk_df = df.filter((df.Attrition == "No") & (df.JobSatisfaction < 3)) \
    .select("Employeenumber", "Age", "Department", "JobRole", "JobSatisfaction", "YearsAtCompany", "Gender", "MaritalStatus")

high_risk_df.write.format("delta").mode("overwrite").saveAsTable("`external-catalog`.default.high_risk_attrition_employees")


In [0]:
l

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "`external-catalog`.default.high_risk_attrition_employees")
history_df = delta_table.history()
display(history_df.select("version", "timestamp", "operation", "userName"))

In [0]:
high_risk_df = spark.read.table("`external-catalog`.default.high_risk_attrition_employees")
display(high_risk_df)

In [0]:
from pyspark.sql import Row

dummy_record = [Row(Employeenumber=99999, Age=30, Department="Sales", JobRole="Sales Executive", JobSatisfaction=1, YearsAtCompany=0, Gender="M", MaritalStatus="Single")]
dummy_df = spark.createDataFrame(dummy_record)

dummy_df.write.format("delta").mode("append").saveAsTable("`external-catalog`.default.high_risk_attrition_employees")

In [0]:
high_risk_df = spark.read.table("`external-catalog`.default.high_risk_attrition_employees")
display(high_risk_df)


In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "`external-catalog`.default.high_risk_attrition_employees")
history_df = delta_table.history()
display(history_df.select("version", "timestamp", "userName", "operation"))

In [0]:
df_v1 = spark.read.format("delta").option("versionAsOf", 1).table("`external-catalog`.default.high_risk_attrition_employees")
display(df_v1)

In [0]:
df_v0 = spark.read.format("delta").option("versionAsOf", 0).table("`external-catalog`.default.high_risk_attrition_employees")
display(df_v0)

In [0]:
df_v2 = spark.read.format("delta").option("versionAsOf", 2).table("`external-catalog`.default.high_risk_attrition_employees")
display(df_v2)

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "`external-catalog`.default.high_risk_attrition_employees")

# Example: Update JobSatisfaction to 5 for Employeenumber 99999
delta_table.update(
    condition="Employeenumber = 99999",
    set={"JobSatisfaction": "5"}
)

In [0]:
df_v3 = spark.read.format("delta").option("versionAsOf", 3).table("`external-catalog`.default.high_risk_attrition_employees")
display(df_v3)

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "`external-catalog`.default.high_risk_attrition_employees")
history_df = delta_table.history()  # full history

display(history_df.select("version", "timestamp", "operation", "operationParameters", "userName", "isBlindAppend", "isolationLevel", "operationMetrics", "userMetadata"))

In [0]:
spark.sql("""
CREATE VOLUME `external-catalog`.default.employee_transformed_managed_volume
""")

In [0]:
from pyspark.sql.functions import when

# Logical transformation: Flag high risk if JobSatisfaction < 3 and YearsAtCompany < 2
transformed_df = df.withColumn(
    "HighRiskFlag",
    when((df.JobSatisfaction < 3) & (df.YearsAtCompany < 2), 1).otherwise(0)
)

transformed_df.write.mode("overwrite") \
    .partitionBy("Department") \
    .format("delta") \
    .save("/Volumes/external-catalog/default/employee_transformed_managed_volume")